# 01 · Sample & Chunk — `scale_15K` (15,000 papers)
Platform: Kaggle T4

In [ ]:
# ══════════════════════════════════════════════════════════════
# SCALE EXPERIMENT CONFIG  ← only N_PAPERS and SCALE_LABEL change between tiers
# ══════════════════════════════════════════════════════════════
import os, sys, json, random
import numpy as np
import pandas as pd
import torch

SCALE_LABEL   = "scale_15K"        # identifier used in output filenames
N_PAPERS      = 15000              # ← THE ONLY NUMBER THAT CHANGES PER TIER
RANDOM_SEED   = 42               # NEVER change this
CHUNK_SIZE    = 400              # tokens
CHUNK_OVERLAP = 50               # tokens
RRF_K         = 60               # standard RRF parameter
TOP_K_STAGE1  = 50               # candidates sent to reranker
EVAL_K_VALUES = [1, 3, 5, 10, 20]

# ── Hardware (fixed by FIX 1) ─────────────────────────────────────────────────
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)

# ── Paths ─────────────────────────────────────────────────────────────────────
DATA_ROOT    = "../../1_data"
RESULTS_DIR  = f"../../4_results/{SCALE_LABEL}"
GT_PATH      = f"{DATA_ROOT}/eval/ground_truth.json"
os.makedirs(RESULTS_DIR, exist_ok=True)

print(f"Scale : {SCALE_LABEL} | N = {N_PAPERS:,} | Device: {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU   : {torch.cuda.get_device_name(0)} | VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

In [ ]:
# Load full CORD-19 metadata
meta_path = f"{DATA_ROOT}/raw/metadata.csv"
df_meta = pd.read_csv(meta_path, low_memory=False)
df_meta = df_meta[df_meta['abstract'].notna()].copy()
print(f"Papers with abstract: {len(df_meta):,}")

In [ ]:
# Stratified sample by publication year
df_meta['year'] = pd.to_datetime(df_meta['publish_time'], errors='coerce').dt.year
df_meta = df_meta[df_meta['year'].notna()].copy()

year_counts = df_meta['year'].value_counts()
year_weights = (year_counts / year_counts.sum())

sampled_ids = []
for year, weight in year_weights.items():
    n_year = max(1, int(round(N_PAPERS * weight)))
    pool = df_meta[df_meta['year'] == year]['cord_uid'].tolist()
    n_year = min(n_year, len(pool))
    sampled = random.sample(pool, n_year)
    sampled_ids.extend(sampled)

# Trim to exactly N_PAPERS
random.shuffle(sampled_ids)
sampled_ids = sampled_ids[:N_PAPERS]
df_sample = df_meta[df_meta['cord_uid'].isin(sampled_ids)].reset_index(drop=True)
print(f"Sampled: {len(df_sample):,} papers (target: {N_PAPERS:,})")
print(df_sample['year'].value_counts().sort_index().head(10))

In [ ]:
# Sentence-window chunking (winner from General/02_chunking_strategy.ipynb)
import re

def sentence_window_chunk(text, chunk_size=CHUNK_SIZE, overlap=CHUNK_OVERLAP):
    """Split text into overlapping sentence-boundary chunks."""
    sentences = re.split(r'(?<=[.!?])\s+', text.strip())
    chunks, current, current_len = [], [], 0
    for sent in sentences:
        tokens = sent.split()
        if current_len + len(tokens) > chunk_size and current:
            chunks.append(' '.join(current))
            # keep overlap
            overlap_sents = []
            overlap_len = 0
            for s in reversed(current):
                l = len(s.split())
                if overlap_len + l <= overlap:
                    overlap_sents.insert(0, s)
                    overlap_len += l
                else:
                    break
            current = overlap_sents
            current_len = overlap_len
        current.append(sent)
        current_len += len(tokens)
    if current:
        chunks.append(' '.join(current))
    return chunks

rows = []
for _, paper in df_sample.iterrows():
    text = f"{paper.get('title','')} {paper.get('abstract','')}".strip()
    if not text:
        continue
    chunks = sentence_window_chunk(text)
    for i, chunk in enumerate(chunks):
        rows.append({
            'cord_uid': paper['cord_uid'],
            'chunk_id': f"{paper['cord_uid']}_{i}",
            'text': chunk,
            'title': paper.get('title',''),
            'year': paper.get('year', None),
            'token_count': len(chunk.split())
        })

df_chunks = pd.DataFrame(rows)
print(f"Chunks: {len(df_chunks):,} from {df_sample.cord_uid.nunique():,} papers")
print(f"Mean tokens: {df_chunks['token_count'].mean():.1f}")
print(f"p95 tokens : {df_chunks['token_count'].quantile(0.95):.0f}")

In [ ]:
# Save chunks
out_path = f"{RESULTS_DIR}/chunks.parquet"
df_chunks.to_parquet(out_path, index=False)
print(f"Saved: {out_path}  ({len(df_chunks):,} rows)")